# ML Pipeline Tutorial — learning the concepts behind `SPARK_SisFall_ML_Pipeline.ipynb`

**What this is:** a companion to `training/notebooks/SPARK_SisFall_ML_Pipeline.ipynb` — not a replacement for it. That notebook is the actual deliverable; this one teaches *why* each stage does what it does, using the same real SisFall data and the same architecture decisions, so you understand the pipeline well enough to defend it, extend it, or debug it later.

**How to use this:**
- Each of the 6 stages below has: a short concept explainer → a worked example using real SPARK data → an exercise for you to complete → an answer you can check afterward.
- Exercises are marked `# TODO` — fill those in yourself before running. Don't skip to the answer cell first; you learn less that way.
- This notebook shares the same repo clone and mounted Drive as the working pipeline — run its Setup cells first, same as that one.

**Assumed context** (from the working notebook, not re-derived here): the task is binary FALL-vs-NON_FALL classification on SisFall IMU data, matching the same convention `train_cnn.py` uses in the SPARK repo — this tutorial builds understanding for that exact task, not a generic ML tutorial.


## Setup

Identical to the working notebook's Setup — same repo, same mount. If you already ran the working notebook this Colab session, Drive is already mounted and the repo already cloned; these cells will just no-op safely.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/Aaradhya-Dev-Tamrakar/SPARK.git /content/SPARK 2>&1 | tail -5
!pip install -q xgboost scikit-learn seaborn


In [ ]:
import sys
sys.path.insert(0, '/content/SPARK/training/data_prep')

import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid")


---
# Stage 1 — Dataset Preparation

**The concept:** raw sensor data almost never arrives in a shape a model can consume directly. SisFall's raw files are variable-length time-series text dumps — one file per (subject, activity, trial), each running for however long that trial happened to last. A model needs a *fixed-size* input every time. Dataset preparation is the process of turning "however-long raw recordings" into "N examples, each exactly the same shape."

For SPARK specifically, that fixed shape is a **200×6 window**: 200 timesteps (2 seconds at 100 Hz) × 6 channels (3-axis accelerometer + 3-axis gyroscope) — chosen to match the actual on-device buffer size the ESP32-S3 firmware will use, not an arbitrary ML convention. This is why the working notebook calls `prepare_sisfall.py` instead of using a generic windowing library — the window size is a *hardware* constraint, not a modeling choice.

**Key sub-steps inside `prepare_sisfall.py`** (you don't need to re-read the script line by line — this is what it's doing conceptually):
1. Parse each raw `.txt` file into a numeric array.
2. Resample from SisFall's native 200 Hz down to SPARK's 100 Hz (linear interpolation) — because the source dataset and the target hardware don't share a sampling rate.
3. Slice each trial into non-overlapping 200-sample windows.
4. Label each window by its source file's activity code (F01–F15 = fall, D01–D19 = ADL).


In [ ]:
# Worked example — run Stage 1 for real, same as the working notebook.
# If you already ran it this session, this just re-reads the cached output.

SISFALL_ZIP = Path('/content/drive/MyDrive/SisFall_dataset.zip')
SISFALL_SRC = Path('/content/SisFall_dataset')
OUT_DIR = Path('/content/sisfall_prepared')

assert SISFALL_ZIP.is_file(), f"Zip not found at {SISFALL_ZIP} — edit SISFALL_ZIP above."

if not SISFALL_SRC.is_dir():
    import zipfile
    with zipfile.ZipFile(SISFALL_ZIP) as zf:
        zf.extractall('/content')
    if not SISFALL_SRC.is_dir():
        candidates = [p for p in Path('/content').glob('**/SA01') if p.is_dir()]
        if candidates:
            SISFALL_SRC = candidates[0].parent

if not OUT_DIR.exists() or not (OUT_DIR / 'windows.npy').exists():
    result = subprocess.run(
        [sys.executable, '/content/SPARK/training/data_prep/prepare_sisfall.py',
         '--src', str(SISFALL_SRC), '--out', str(OUT_DIR)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("prepare_sisfall.py failed — see stderr above.")

windows = np.load(OUT_DIR / 'windows.npy')
labels_raw = np.load(OUT_DIR / 'labels.npy')

print(f"One raw trial file becomes MANY windows — here's the shape after windowing:")
print(f"  windows.shape = {windows.shape}  ->  ({windows.shape[0]} windows, {windows.shape[1]} timesteps, {windows.shape[2]} channels)")
print(f"  a single window looks like: windows[0].shape = {windows[0].shape}")


### Exercise 1

Using `windows` and `labels_raw` from the cell above:

1. How many total windows came out of the whole dataset?
2. What's the shape of **one single window** — write it as `(timesteps, channels)`.
3. Print the first 5 raw labels (`labels_raw[:5]`) — what format are they in (e.g. "F03", "D11")?

Fill in the `# TODO` lines below.


In [ ]:
# TODO: print the total number of windows
n_total_windows = None  # replace None
print(f"Total windows: {n_total_windows}")

# TODO: print the shape of a single window
single_window_shape = None  # replace None
print(f"Single window shape: {single_window_shape}")

# TODO: print the first 5 raw labels
first_five_labels = None  # replace None
print(f"First 5 labels: {first_five_labels}")


<details>
<summary>Answer (click to expand)</summary>

```python
n_total_windows = windows.shape[0]
single_window_shape = windows[0].shape
first_five_labels = labels_raw[:5]
```

The labels are native SisFall activity codes — `F01`–`F15` for falls, `D01`–`D19` for ADLs (activities of daily living). Note they are **not yet binary** at this stage — that collapse happens explicitly in Stage 2, as a deliberate downstream choice rather than something baked into parsing.
</details>


---
# Stage 2 — Dataset Preprocessing

**The concept:** preprocessing is everything that happens *after* you have clean, fixed-shape data, but *before* a model can learn from it. Two distinct things happen here, and it's worth keeping them mentally separate:

**(a) Feature engineering** — turning a raw 200×6 time-series window into a smaller set of numbers that summarize it. This step exists specifically *because* Random Forest and XGBoost (Stage 4's algorithms) expect **tabular input**: one row, fixed number of columns, no sequence structure. They have no concept of "timestep 47 comes before timestep 48." A neural network (like SPARK's own CNN in `train_cnn.py`) *can* consume the raw sequence directly — which is exactly why that script skips this step entirely. Feature engineering is not a universal ML requirement; it's specifically needed for the algorithm family used in Stage 4.

Common statistical features per channel: mean, standard deviation, min, max, range, RMS (root-mean-square — captures overall signal energy). Plus domain-specific features when you know something about the task — SPARK's notebook adds **peak resultant acceleration**, because a fall physically produces a sharp acceleration spike; that's a feature invented from knowing the problem, not a generic statistic.

**(b) Splitting + scaling** — you can't just randomly shuffle windows into train/test. If windows from the *same subject* end up in both train and test, the model can partly "recognize" that subject's movement signature rather than learning general fall-vs-not-fall patterns — this is **data leakage**, and it makes your test accuracy a lie. The fix is **grouping by subject**: every window from one person goes entirely into one split. Scaling (subtracting mean, dividing by std) also has a subtle trap: you must compute the mean/std from the *training data only*, then apply those same numbers to val/test — never recompute scaling stats on val/test, or you've leaked information about their distribution back into "unseen" data.


In [ ]:
# Worked example — engineer features for just ONE window, so you can see exactly what
# the transformation does before it's applied to all of them at once.

def extract_features_demo(window: np.ndarray) -> dict:
    channel_names = ['a_x', 'a_y', 'a_z', 'w_x', 'w_y', 'w_z']
    feats = {}
    for i, ch in enumerate(channel_names):
        col = window[:, i]
        feats[f'{ch}_mean'] = col.mean()
        feats[f'{ch}_std'] = col.std()
    return feats

one_window = windows[0]
print(f"BEFORE: one window is a {one_window.shape} array — 200 rows, 6 columns, hard to feed to a tree model.")
print()
demo_features = extract_features_demo(one_window)
print(f"AFTER: the same window becomes a flat dict of {len(demo_features)} numbers:")
for k, v in demo_features.items():
    print(f"  {k}: {v:.3f}")


### Exercise 2

The demo above only computed `mean` and `std` per channel. The real pipeline (Stage 2 of the working notebook) also computes `min`, `max`, `range`, and `rms`.

Extend `extract_features_demo` below to also compute **`range`** (max − min) for each channel, and run it on `windows[0]`.


In [ ]:
def extract_features_exercise(window: np.ndarray) -> dict:
    channel_names = ['a_x', 'a_y', 'a_z', 'w_x', 'w_y', 'w_z']
    feats = {}
    for i, ch in enumerate(channel_names):
        col = window[:, i]
        feats[f'{ch}_mean'] = col.mean()
        feats[f'{ch}_std'] = col.std()
        # TODO: add feats[f'{ch}_range'] = ...
    return feats

result = extract_features_exercise(windows[0])
print(result)


<details>
<summary>Answer (click to expand)</summary>

```python
feats[f'{ch}_range'] = col.max() - col.min()
```

Added inside the same per-channel loop, right after `_std`. `range` matters for fall detection specifically because a fall produces a much wider swing between the highest and lowest acceleration values within the 2-second window than an ADL like walking does — it's a cheap, interpretable signal for exactly the pattern you're trying to detect.
</details>


---
# Stage 3 — EDA (Exploratory Data Analysis)

**The concept:** EDA is looking at your data *before* you trust a model's output on it — checking your assumptions hold, catching problems a model would otherwise silently learn around (or fail on). Three checks matter most here:

1. **Class balance** — how many FALL windows vs. NON_FALL windows? SisFall (like most fall datasets) is naturally imbalanced — ADLs vastly outnumber falls in real life, and the dataset reflects that. If you don't check this, you might not notice a model that's 95% "accurate" by just always predicting NON_FALL.
2. **Does the most task-relevant feature actually separate the classes?** If your best domain-informed feature (peak resultant acceleration) shows *no* visible difference between FALL and NON_FALL windows, that's a red flag worth investigating before you even get to modeling — it might mean a labeling bug, a windowing bug, or that the feature genuinely doesn't carry the signal you assumed.
3. **Feature correlation** — are any two engineered features nearly identical? Highly correlated features don't help a model (they're redundant) and can make feature-importance interpretation misleading later.


In [ ]:
# Worked example — real class balance on the actual SPARK data.
labels_bin_demo = np.where(np.char.startswith(labels_raw, 'F'), 1, 0)

n_fall = int(labels_bin_demo.sum())
n_nonfall = int((1 - labels_bin_demo).sum())
fall_ratio = labels_bin_demo.mean()

print(f"FALL windows:     {n_fall}")
print(f"NON_FALL windows: {n_nonfall}")
print(f"Fall ratio:       {fall_ratio:.3f}  ({fall_ratio*100:.1f}% of all windows are FALL)")


### Exercise 3

Given the imbalance you just printed above — this is *why* the modeling stage (next) uses `class_weight='balanced'` for Random Forest and `scale_pos_weight` for XGBoost, instead of training with default settings.

In your own words (edit the markdown cell below, or just think it through), answer: **if you trained a model on this data with no imbalance correction at all, what's the laziest way it could get a deceptively high accuracy score — and why would that model be useless for SPARK's actual purpose?**


_Write your answer here — replace this line._


<details>
<summary>Answer (click to expand)</summary>

With no correction, the easiest path to high *accuracy* is to just predict NON_FALL for everything. If NON_FALL windows are the large majority, that strategy alone can score deceptively well on raw accuracy — while catching **zero** actual falls. That model would have 0% Sensitivity (Recall on the FALL class) despite looking "accurate" — and Sensitivity is exactly the metric SPARK's proposal sets a ≥90% target for, because a fall-detection system that misses falls is worse than useless; it's a false sense of safety. This is precisely why the working notebook reports Sensitivity and Specificity *separately*, instead of a single accuracy number — a single number can hide this failure mode completely.
</details>


---
# Stage 4 — Modeling (Random Forest & XGBoost, hyperparameter-tuned)

**The concept, in two parts:**

**(a) Why two different algorithms?** Random Forest builds many decision trees independently on random subsets of data/features and averages their votes — each tree can be "wrong" in a different way, and averaging cancels out a lot of that noise. XGBoost builds trees **sequentially**, where each new tree specifically tries to correct the mistakes of the trees before it (gradient boosting). Both are tree-based, both handle tabular data well, but they fail differently and tune differently — which is exactly why using both gives you a more trustworthy comparison than trusting either one alone. This is also the concrete reason the assignment asks for "at least 2 algorithms," not "the best algorithm you can find" — comparing outputs on the *same* engineered features is itself informative.

**(b) What is hyperparameter tuning, concretely?** A hyperparameter is a setting you choose *before* training (not something the model learns from data) — like how many trees to grow, or how deep each tree is allowed to go. `GridSearchCV` tries every combination in a specified grid, scores each combination via cross-validation, and keeps the best-scoring combination. **Cross-validation** itself means: split the training data into K folds, train on K−1 of them, validate on the 1 left out, rotate which fold is held out, and average the scores — this gives a much more reliable estimate than a single train/validation split, because you're not depending on one lucky (or unlucky) split.

One more detail specific to this pipeline: the cross-validation folds themselves are **grouped by subject** (`GroupKFold`), for the exact same leakage reason as Stage 2's train/test split — tuning on leaky folds would pick hyperparameters that look great on paper but don't generalize to a person the model has never seen.


In [ ]:
# Worked example — a TINY grid search on real SPARK features, small enough to run in seconds,
# so you can see exactly what GridSearchCV is doing before the real pipeline runs its full grid.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# Quick feature table (same extraction as Stage 2, condensed for this demo)
def extract_features(window):
    channel_names = ['a_x', 'a_y', 'a_z', 'w_x', 'w_y', 'w_z']
    feats = {}
    for i, ch in enumerate(channel_names):
        col = window[:, i]
        feats[f'{ch}_mean'] = col.mean()
        feats[f'{ch}_std'] = col.std()
        feats[f'{ch}_range'] = col.max() - col.min()
    return feats

meta_demo = pd.read_csv(OUT_DIR / 'meta.csv')
feature_rows = [extract_features(w) for w in windows]
X_demo = pd.DataFrame(feature_rows)
y_demo = np.where(np.char.startswith(labels_raw, 'F'), 1, 0)
subjects_demo = meta_demo['subject'].values

X_train_demo, _, y_train_demo, _ = X_demo.values, None, y_demo, None
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=RANDOM_SEED)
train_idx_demo, _ = next(gss.split(X_demo, y_demo, groups=subjects_demo))

# A TINY grid — just 2 combinations — vs. the real notebook's larger grid, so this runs fast.
tiny_grid = {'n_estimators': [50, 100], 'max_depth': [5]}

grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_SEED, class_weight='balanced'),
    tiny_grid,
    cv=GroupKFold(n_splits=3).split(X_demo.values[train_idx_demo], y_demo[train_idx_demo], groups=subjects_demo[train_idx_demo]),
    scoring='f1',
)
grid.fit(X_demo.values[train_idx_demo], y_demo[train_idx_demo])

print("Every combination GridSearchCV tried, and its cross-validated F1 score:")
for params, mean_score in zip(grid.cv_results_['params'], grid.cv_results_['mean_test_score']):
    print(f"  {params}  ->  F1 = {mean_score:.4f}")
print()
print(f"Best combination chosen: {grid.best_params_}  (F1 = {grid.best_score_:.4f})")


### Exercise 4

The real working notebook's grid for Random Forest is:
```python
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 16, None],
    'min_samples_leaf': [1, 4],
}
```

**Without running it** — just by counting — how many total hyperparameter combinations does `GridSearchCV` try for this grid? (Hint: multiply the number of options in each list together.)


In [ ]:
# TODO: compute the total number of combinations by multiplying list lengths
n_n_estimators = None    # how many options in 'n_estimators'?
n_max_depth = None       # how many options in 'max_depth'?
n_min_samples_leaf = None  # how many options in 'min_samples_leaf'?

total_combinations = None  # TODO: multiply the three above together
print(f"Total combinations GridSearchCV will try: {total_combinations}")


<details>
<summary>Answer (click to expand)</summary>

```python
n_n_estimators = 2       # [100, 200]
n_max_depth = 3          # [8, 16, None]
n_min_samples_leaf = 2   # [1, 4]
total_combinations = 2 * 3 * 2  # = 12
```

12 combinations, each evaluated across 5 cross-validation folds (per the real notebook's `GroupKFold(n_splits=5)`) — so Random Forest tuning alone trains **60 separate models** before picking the single best one. This is why grid search can be slow: it scales multiplicatively with every hyperparameter you add options for.
</details>


---
# Stage 5 — Evaluation

**The concept:** once you have a trained, tuned model, you need to know how good it actually is — on data it has genuinely never seen (the held-out test set, untouched by any tuning). Accuracy alone is misleading for imbalanced problems (see Stage 3's exercise) — so this pipeline reports four metrics, each answering a different question:

- **Sensitivity** (a.k.a. Recall on the FALL class): *of all the actual falls, how many did the model catch?* This is the single most safety-critical number for SPARK — miss a fall, and the system has failed at its entire purpose.
- **Specificity** (Recall on the NON_FALL class): *of all the actual non-falls, how many did the model correctly leave alone?* Low specificity means constant false alarms — annoying, and erodes trust in the system over time.
- **F1-score**: the harmonic mean of precision and recall — a single balanced number, useful for comparing models at a glance, but it can hide which of Sensitivity/Specificity is doing the heavy lifting, which is why it's reported *alongside* them, not instead of them.
- **AUC-ROC**: measures how well the model ranks FALL windows above NON_FALL windows across *every possible decision threshold*, not just the default 50% cutoff — useful because in deployment you might want to tune that threshold (e.g. bias toward higher Sensitivity even at some Specificity cost, given the safety stakes).

Notice these are exactly the same four metrics `train_cnn.py` reports for the CNN — deliberately, so RF/XGB and CNN results are directly comparable, not measured on different yardsticks.


In [ ]:
# Worked example — confusion matrix concept, illustrated on the tiny model from Stage 4's demo,
# evaluated on a slice of data it wasn't trained on (mimicking held-out evaluation).

heldout_idx = np.setdiff1d(np.arange(len(X_demo)), train_idx_demo)[:500]  # small slice for a fast demo
X_heldout = X_demo.values[heldout_idx]
y_heldout = y_demo[heldout_idx]

y_pred_demo = grid.predict(X_heldout)

fall_mask = y_heldout == 1
nonfall_mask = y_heldout == 0

sensitivity_demo = (y_pred_demo[fall_mask] == 1).sum() / fall_mask.sum() if fall_mask.sum() else float('nan')
specificity_demo = (y_pred_demo[nonfall_mask] == 0).sum() / nonfall_mask.sum() if nonfall_mask.sum() else float('nan')

print(f"On this small held-out slice:")
print(f"  Sensitivity (caught {int((y_pred_demo[fall_mask]==1).sum())} of {int(fall_mask.sum())} actual falls): {sensitivity_demo:.3f}")
print(f"  Specificity (correctly cleared {int((y_pred_demo[nonfall_mask]==0).sum())} of {int(nonfall_mask.sum())} actual non-falls): {specificity_demo:.3f}")
print()
print("(This is a demo on a tiny 2-combination grid + small slice — expect noisier numbers than the real pipeline's full run.)")


### Exercise 5

Given Sensitivity and Specificity printed above — write out, in plain language, what a **false negative** means for SPARK specifically (not the general statistics definition — the real-world consequence for this exact system), and which of the two metrics above would drop if false negatives increased.


_Write your answer here — replace this line._


<details>
<summary>Answer (click to expand)</summary>

A false negative here means: **a real fall happened, and the model classified it as NON_FALL** — the system stays silent when someone has actually fallen. For SPARK, that's the single worst failure mode: the caregiver never gets alerted, and the whole point of the wearable is defeated at the exact moment it matters most. An increase in false negatives directly **lowers Sensitivity** — because Sensitivity is defined as (true positives) / (all actual positives), and every false negative is an actual positive the model failed to catch. This is also why the proposal sets a hard ≥90% Sensitivity target rather than just aiming for good overall accuracy.
</details>


---
# Stage 6 — Predictions

**The concept:** this is the payoff stage — taking your trained, evaluated model and actually using it to classify new examples, one at a time, the way it would be used in a real deployment (a single window comes in, the model outputs a verdict). Two things worth understanding here:

**(a) Predicted probability vs. predicted class.** Most classifiers can output either a hard class label (`0` or `1`) *or* a probability (e.g. "73% confident this is a FALL"). The probability is strictly more informative — it tells you *how* confident the model was, which matters a lot for a safety system. A model that's 51% confident and a model that's 99% confident might both output the same hard label, but you'd want to treat those very differently in a real alert pipeline (e.g. maybe 51% confidence triggers a soft notification, 99% triggers an urgent one).

**(b) Why compare true label vs. predicted label per-example, not just in aggregate?** Aggregate metrics (Stage 5) tell you the model's *overall* behavior, but looking at individual predictions — especially the ones it got wrong — is often how you discover *why* a model fails, which aggregate numbers alone never show you. This is standard practice before trusting any model with real consequences.


In [ ]:
# Worked example — run the tiny demo model from Stage 4 on a handful of individual windows,
# and look at true label vs. predicted label side-by-side.

sample_idx_demo = np.random.RandomState(RANDOM_SEED).choice(len(X_heldout), size=5, replace=False)

preds_demo = grid.predict(X_heldout[sample_idx_demo])
probs_demo = grid.predict_proba(X_heldout[sample_idx_demo])[:, 1]
true_demo = y_heldout[sample_idx_demo]

demo_table = pd.DataFrame({
    'true_label': np.where(true_demo == 1, 'FALL', 'NON_FALL'),
    'predicted': np.where(preds_demo == 1, 'FALL', 'NON_FALL'),
    'fall_probability': probs_demo.round(3),
    'correct': (preds_demo == true_demo),
})
demo_table


### Exercise 6

Look at the table above. Pick **one row where `correct` is `False`** (if none appear in your random sample, re-run the cell above a couple of times, or read this as a hypothetical using the columns' meaning).

Write one sentence: given that row's `fall_probability`, does it look like a **confident wrong guess** (probability far from 50%) or an **uncertain wrong guess** (probability close to 50%) — and why would that distinction matter if this were a real deployed alert system?


_Write your answer here — replace this line._


<details>
<summary>Answer (click to expand)</summary>

There's no single fixed answer here — it depends on which row you happened to get — but the reasoning pattern is: a wrong prediction with `fall_probability` near 0.5 means the model was genuinely uncertain and just landed on the wrong side of the 50% cutoff by a small margin. A wrong prediction with `fall_probability` near 0.0 or 1.0 means the model was **confidently wrong** — which is more concerning, because it suggests the model has learned something systematically misleading about that kind of example, not just noise near a decision boundary. In a real deployed system, this distinction is exactly why you'd expose the probability (not just the hard label) to the alerting logic — an uncertain case might warrant a lower-urgency notification or a request for more data, while a confidently-wrong pattern is worth going back and investigating in the training data itself.
</details>


---
## Recap

You've now worked through, on real SPARK data, the same six stages the working notebook (`SPARK_SisFall_ML_Pipeline.ipynb`) runs at full scale:

1. **Dataset preparation** — raw variable-length sensor files → fixed-size windows, because both the model and the target hardware need a consistent shape.
2. **Preprocessing** — raw windows → engineered tabular features (for tree models specifically), split and scaled without leaking information across subjects.
3. **EDA** — checking class balance and feature relevance *before* trusting a model's output on the data.
4. **Modeling** — two different tree-based algorithms, each hyperparameter-tuned via grouped cross-validation, so the comparison itself is informative.
5. **Evaluation** — four complementary metrics on genuinely held-out data, chosen because accuracy alone hides the exact failure mode (missed falls) that matters most here.
6. **Predictions** — turning a trained model into individual verdicts, and treating the probability, not just the hard label, as the real output worth inspecting.

**Next step:** go run the full working notebook (`training/notebooks/SPARK_SisFall_ML_Pipeline.ipynb`) — everything above should now read as familiar rather than as new material to decode.
